# Evaluación Parcial 2: Machine Learning supervisado

**Asignatura:** MLY0100 | **Integrantes:** Christian Sandoval y Nicolás Vega | **Fecha:** 5 de noviembre de 2026

**Dataset:** Properati limpio de EV1. **CRISP-DM:** fases 4 (Modelado) y 5 (Evaluación).

Se desarrollan dos tareas: regresión para estimar price en USD y clasificación de property_type en Departamento, Casa, PH y Otros. Las conclusiones se generan a partir de las métricas ejecutadas.

## 1. Contexto y objetivos

El caso estudia publicaciones de propiedades en venta de Argentina. En regresión se estima un valor monetario continuo (price); en clasificación se asigna una categoría discreta (property_type). Se comparan modelos supervisados y su generalización. property_type solo se usa como feature en regresión. title y description se excluyen de clasificación para evitar data leakage.

## 2. Librerías

Se utilizan Pandas, NumPy, Matplotlib, Seaborn, Scikit-Learn e imbalanced-learn para preparar, modelar y evaluar los datos.

In [ ]:
# Importamos las librerías que utilizaremos
import os
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
from imblearn.over_sampling import RandomOverSampler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
pd.set_option('display.max_columns', 40)

print('Librerías cargadas correctamente.')

## 3. Carga del dataset limpio de EV1

Se busca el CSV preparado. Si no existe, se reconstruye desde el ZIP aplicando la limpieza final de EV1: medianas agrupadas e identificación conjunta de outliers mediante IQR y Z-score. Se conservan variables originales; los escaladores se ajustan nuevamente después de la división train/test para evitar fuga de información.

In [ ]:
COLUMNAS_MODELO = [
    'lat', 'lon', 'l2', 'l3', 'rooms', 'bedrooms', 'bathrooms',
    'surface_total', 'surface_covered', 'price', 'property_type'
]
COLUMNAS_NUMERICAS_EV1 = [
    'rooms', 'bedrooms', 'bathrooms', 'surface_total',
    'surface_covered', 'price', 'lat', 'lon'
]


def localizar_archivo(nombre):
    candidatos = [
        Path.cwd() / nombre,
        Path.cwd().parent / nombre,
        Path('/content') / nombre,
        Path('/content/drive/MyDrive') / nombre
    ]
    for ruta in candidatos:
        if ruta.exists():
            return ruta
    return None


ruta_preparado = localizar_archivo('DS1-18-Datos-Properati-preparado.csv')
ruta_zip = localizar_archivo('Evaluación Parcial 1 - Dataset.zip')

if ruta_preparado is not None:
    print(f'Leyendo dataset preparado de EV1: {ruta_preparado}')
    df = pd.read_csv(ruta_preparado, usecols=lambda c: c in COLUMNAS_MODELO, low_memory=False)
else:
    if ruta_zip is None:
        raise FileNotFoundError(
            'No se encontró el CSV preparado ni el ZIP original. '
            'Coloca "Evaluación Parcial 1 - Dataset.zip" en la raíz del repositorio '
            'o ejecuta primero notebook/evaluacion_parcial_1.ipynb.'
        )
    print(f'No existe el CSV preparado. Se reconstruye la limpieza final de EV1 desde: {ruta_zip}')
    with zipfile.ZipFile(ruta_zip) as archivo_zip:
        nombre_csv = next(n for n in archivo_zip.namelist() if n.endswith('DS1-18-Datos-Properati.csv'))
        with archivo_zip.open(nombre_csv) as archivo_csv:
            df = pd.read_csv(archivo_csv, usecols=COLUMNAS_MODELO, low_memory=False)

    # Imputación final utilizada en EV1: medianas por tipo de propiedad y localidad.
    for columna in ['surface_total', 'surface_covered', 'bathrooms']:
        mediana_grupo = df.groupby('property_type', observed=False)[columna].transform('median')
        df[columna] = df[columna].fillna(mediana_grupo)

    for columna in ['lat', 'lon']:
        mediana_grupo = df.groupby('l3', observed=False)[columna].transform('median')
        df[columna] = df[columna].fillna(mediana_grupo)

    # Criterio de EV1: eliminar solo los registros marcados por IQR y Z-score a la vez.
    mascara_extremos = np.zeros(len(df), dtype=bool)
    for columna in COLUMNAS_NUMERICAS_EV1:
        q1 = df[columna].quantile(0.25)
        q3 = df[columna].quantile(0.75)
        iqr = q3 - q1
        mascara_iqr = ((df[columna] < q1 - 1.5 * iqr) | (df[columna] > q3 + 1.5 * iqr)).to_numpy()
        z_score = np.abs(stats.zscore(df[columna].to_numpy(), nan_policy='omit'))
        mascara_z = z_score > 3
        mascara_extremos |= mascara_iqr & mascara_z
    df = df.loc[~mascara_extremos].copy()

# Tipos consistentes para el análisis y el preprocesamiento.
for columna in ['l2', 'l3', 'property_type']:
    df[columna] = df[columna].astype('string').fillna('Sin dato').astype(str)

for columna in ['lat', 'lon', 'rooms', 'bedrooms', 'bathrooms', 'surface_total', 'surface_covered', 'price']:
    df[columna] = pd.to_numeric(df[columna], errors='coerce')

df = df.dropna(subset=COLUMNAS_MODELO).copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=COLUMNAS_MODELO).copy()

print('Dimensiones del dataset utilizado:', df.shape)
print('Valores faltantes en columnas de modelado:', int(df[COLUMNAS_MODELO].isna().sum().sum()))
print('Tipos de propiedad originales:')
display(df['property_type'].value_counts().to_frame('cantidad'))

## 4. Targets y distribución

price se modela con log1p por su asimetría y las métricas se devuelven a USD. Para clasificación se agrupan Lote, Oficina, Local comercial, Casa de campo, Depósito, Cochera y Otro en Otros.

In [ ]:
df['clase'] = np.where(
    df['property_type'].isin(['Departamento', 'Casa', 'PH']),
    df['property_type'],
    'Otros'
)

print('Target de regresión:')
print('dtype:', df['price'].dtype)
print('Cantidad de valores únicos:', df['price'].nunique())
print(df['price'].describe())

print('\nTarget de clasificación:')
print('dtype:', df['clase'].dtype)
print('Cantidad de clases:', df['clase'].nunique())
display(df['clase'].value_counts().to_frame('cantidad'))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(df['price'], bins=50, edgecolor='black')
ax[0].set_title('Distribución del precio')
ax[0].set_xlabel('Precio (USD)')
ax[0].set_ylabel('Frecuencia')
ax[1].hist(np.log1p(df['price']), bins=50, edgecolor='black')
ax[1].set_title('Distribución de log1p(price)')
ax[1].set_xlabel('log1p(Precio)')
ax[1].set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

frecuencias = df['clase'].value_counts()
porcentajes = (df['clase'].value_counts(normalize=True) * 100).round(2)
display(pd.DataFrame({'cantidad': frecuencias, 'porcentaje': porcentajes}))

plt.figure(figsize=(8, 4))
plt.bar(frecuencias.index.astype(str), frecuencias.values)
plt.title('Distribución de las cuatro clases')
plt.xlabel('Tipo de propiedad agrupado')
plt.ylabel('Cantidad de publicaciones')
plt.xticks(rotation=15)
plt.show()

## 5. División train/test

Se utilizan muestras reproducibles de hasta 12.000 registros para regresión y 8.000 para clasificación. La muestra de clasificación y su división 80/20 son estratificadas. Los cuatro modelos usan la misma división.

In [ ]:
# REGRESIÓN: muestra aleatoria reproducible y división 80/20.
N_REGRESION = min(12_000, len(df))
df_reg = df.sample(n=N_REGRESION, random_state=RANDOM_STATE).copy()

features_reg_numericas = ['rooms', 'bathrooms', 'surface_total', 'surface_covered', 'lat', 'lon']
features_reg_categoricas = ['l2', 'l3', 'property_type']
features_reg = features_reg_numericas + features_reg_categoricas
X_reg = df_reg[features_reg].copy()
y_reg = df_reg['price'].astype(float).copy()

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_STATE
)

# CLASIFICACIÓN: muestra estratificada de hasta 8.000 registros.
N_CLASIFICACION = min(8_000, len(df))
df_clf, _ = train_test_split(
    df, train_size=N_CLASIFICACION, stratify=df['clase'], random_state=RANDOM_STATE
)

features_clf_numericas = ['rooms', 'bathrooms', 'surface_total', 'surface_covered', 'lat', 'lon']
features_clf_categoricas = ['l2', 'l3']
features_clf = features_clf_numericas + features_clf_categoricas
X_clf = df_clf[features_clf].copy()
y_clf = df_clf['clase'].copy()

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_STATE, stratify=y_clf
)

print('Regresión | train:', X_train_reg.shape, '| test:', X_test_reg.shape)
print('Clasificación | train:', X_train_clf.shape, '| test:', X_test_clf.shape)
print('\nDistribución de clasificación en entrenamiento:')
display(y_train_clf.value_counts().to_frame('cantidad_train'))
print('Distribución de clasificación en prueba:')
display(y_test_clf.value_counts().to_frame('cantidad_test'))

## 6. Correlación

La correlación se analiza en entrenamiento. Pearson resume relaciones lineales y Spearman relaciones monótonas. Se revisan pares con |r| > 0,8 y la relación de los atributos numéricos con el precio.

In [ ]:
# bedrooms se incorpora al análisis exploratorio para justificar su descarte,
# pero no se entrega a los modelos porque presenta alta correlación con rooms.
features_corr_numericas = [
    'rooms', 'bedrooms', 'bathrooms', 'surface_total',
    'surface_covered', 'lat', 'lon'
]
df_corr_train = df_reg.loc[X_train_reg.index, features_corr_numericas].copy()
df_corr_train['price'] = y_train_reg

corr_pearson = df_corr_train.corr(method='pearson')
corr_spearman = df_corr_train.corr(method='spearman')

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(corr_pearson, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax[0])
ax[0].set_title('Correlación de Pearson (entrenamiento)')
sns.heatmap(corr_spearman, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax[1])
ax[1].set_title('Correlación de Spearman (entrenamiento)')
plt.tight_layout()
plt.show()

print('Correlación entre rooms y bedrooms en el conjunto de entrenamiento:')
print(f"Pearson: {corr_pearson.loc['rooms', 'bedrooms']:.3f}")
print(f"Spearman: {corr_spearman.loc['rooms', 'bedrooms']:.3f}")

correlacion_precio = corr_spearman['price'].drop('price').sort_values(key=np.abs, ascending=False)
print('Correlación de Spearman de cada variable numérica con price:')
display(correlacion_precio.to_frame('spearman_con_price'))

pares_fuertes = []
for i, variable_a in enumerate(features_corr_numericas):
    for variable_b in features_corr_numericas[i + 1:]:
        valor = corr_pearson.loc[variable_a, variable_b]
        if abs(valor) > 0.8:
            pares_fuertes.append((variable_a, variable_b, valor))

if pares_fuertes:
    display(pd.DataFrame(pares_fuertes, columns=['feature_1', 'feature_2', 'pearson']).sort_values('pearson'))
else:
    print('No se encontraron pares con |Pearson| > 0,8 entre las variables analizadas.')

# Para clasificación también se revisan las medianas por clase, incluyendo
# bedrooms como variable de diagnóstico, aunque no se usa para entrenar.
df_clf_train_eda = X_train_clf.copy()
df_clf_train_eda['bedrooms'] = df_clf.loc[X_train_clf.index, 'bedrooms']
df_clf_train_eda['clase'] = y_train_clf
features_clf_eda = features_clf_numericas + ['bedrooms']
medianas_por_clase = df_clf_train_eda.groupby('clase')[features_clf_eda].median().round(2)
display(medianas_por_clase)

plt.figure(figsize=(9, 5))
sns.boxplot(data=df_clf_train_eda, x='clase', y='surface_total', showfliers=False)
plt.title('Superficie total por clase de propiedad (entrenamiento)')
plt.xlabel('Clase')
plt.ylabel('Superficie total (m²)')
plt.show()

### Decisiones de selección de features

- Se conserva **rooms** y se descarta **bedrooms** en regresión y clasificación para reducir multicolinealidad. En el conjunto de entrenamiento, Pearson entre ambas es aproximadamente **0,887** y Spearman **0,921**. Son asociaciones altas y describen dimensiones cercanas del tamaño de la vivienda.
- Se conservan **surface_total** y **surface_covered**: describen superficie total y cubierta. En la matriz de entrenamiento no superan entre sí el umbral de referencia |r| > 0,8, por lo que aportan información complementaria.
- Se conservan **bathrooms**, **lat** y **lon** por su relación con características físicas y ubicación. Pearson y Spearman permiten observar relaciones lineales y monótonas con price.
- **l2** y **l3** se codifican con One-Hot Encoding. Son variables nominales, por lo que no corresponde asignarles un orden numérico artificial.
- Para regresión se incluye **property_type**, porque el tipo de inmueble es conocido al estimar su precio. Para clasificación se excluye porque es la variable objetivo.
- Se descartan **title** y **description** para clasificación por riesgo de data leakage; también se descartan fechas sin transformar, columnas constantes (l1, currency, operation_type) y columnas escaladas en EV1, que se ajustaron antes de separar train/test.
- **price_minmax** se excluye siempre de los features porque es una transformación directa del target price.

Estas decisiones reducen redundancia y evitan entregar al modelo información que no estaría disponible de forma legítima al realizar una predicción.

## 7. Preparación de los datos para regresión

Primero separamos las variables categóricas y las convertimos a números usando One-Hot Encoding. Después normalizamos los datos con StandardScaler. El escalador se ajusta con los datos de entrenamiento y luego se aplica a los datos de prueba.

In [ ]:
# Aplicamos One-Hot Encoding a las variables categóricas
X_train_reg_dummies = pd.get_dummies(
    X_train_reg,
    columns=features_reg_categoricas,
    drop_first=True
)

X_test_reg_dummies = pd.get_dummies(
    X_test_reg,
    columns=features_reg_categoricas,
    drop_first=True
)

# Dejamos las mismas columnas en entrenamiento y prueba
X_test_reg_dummies = X_test_reg_dummies.reindex(
    columns=X_train_reg_dummies.columns,
    fill_value=0
)

print('Dimensión de X_train:', X_train_reg_dummies.shape)
print('Dimensión de X_test:', X_test_reg_dummies.shape)

# Normalizamos los datos
scaler_reg = StandardScaler()

X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg_dummies)
X_test_reg_scaled = scaler_reg.transform(X_test_reg_dummies)

print('Datos de regresión preparados correctamente.')

## 8. Modelos de regresión

Implementaremos dos modelos vistos en la asignatura:

- **Regresión lineal múltiple:** se utiliza como modelo base para estimar el precio a partir de varias características.
- **KNN Regressor:** estima el precio utilizando las propiedades más cercanas según sus características. Por trabajar con distancias, necesita que los atributos estén normalizados.

La variable que queremos predecir es price, expresada en dólares.

In [ ]:
# Creamos el modelo de Regresión Lineal
modelo_lineal = LinearRegression()

# Entrenamos el modelo
modelo_lineal.fit(X_train_reg_scaled, y_train_reg)

# Realizamos las predicciones
pred_lineal_train = modelo_lineal.predict(X_train_reg_scaled)
pred_lineal_test = modelo_lineal.predict(X_test_reg_scaled)

# Creamos el modelo KNN
modelo_knn_reg = KNeighborsRegressor(n_neighbors=7)

# Entrenamos el modelo
modelo_knn_reg.fit(X_train_reg_scaled, y_train_reg)

# Realizamos las predicciones
pred_knn_train = modelo_knn_reg.predict(X_train_reg_scaled)
pred_knn_test = modelo_knn_reg.predict(X_test_reg_scaled)

print('Modelos de regresión entrenados correctamente.')

## 9. Evaluación de los modelos de regresión

Utilizaremos tres métricas:

- **MAE:** indica el error absoluto promedio en dólares.
- **RMSE:** penaliza más los errores grandes.
- **R²:** indica qué proporción de la variación del precio explica el modelo.

Calcularemos las métricas para entrenamiento y prueba, como en los ejercicios de clases.

In [ ]:
# Calculamos las métricas de Regresión Lineal
mae_lineal_train = mean_absolute_error(y_train_reg, pred_lineal_train)
mae_lineal_test = mean_absolute_error(y_test_reg, pred_lineal_test)

rmse_lineal_train = np.sqrt(mean_squared_error(y_train_reg, pred_lineal_train))
rmse_lineal_test = np.sqrt(mean_squared_error(y_test_reg, pred_lineal_test))

r2_lineal_train = r2_score(y_train_reg, pred_lineal_train)
r2_lineal_test = r2_score(y_test_reg, pred_lineal_test)

# Calculamos las métricas de KNN
mae_knn_train = mean_absolute_error(y_train_reg, pred_knn_train)
mae_knn_test = mean_absolute_error(y_test_reg, pred_knn_test)

rmse_knn_train = np.sqrt(mean_squared_error(y_train_reg, pred_knn_train))
rmse_knn_test = np.sqrt(mean_squared_error(y_test_reg, pred_knn_test))

r2_knn_train = r2_score(y_train_reg, pred_knn_train)
r2_knn_test = r2_score(y_test_reg, pred_knn_test)

# Creamos una tabla para comparar los resultados
tabla_regresion = pd.DataFrame({
    'MAE_train': [mae_lineal_train, mae_knn_train],
    'MAE_test': [mae_lineal_test, mae_knn_test],
    'RMSE_train': [rmse_lineal_train, rmse_knn_train],
    'RMSE_test': [rmse_lineal_test, rmse_knn_test],
    'R2_train': [r2_lineal_train, r2_knn_train],
    'R2_test': [r2_lineal_test, r2_knn_test]
}, index=['Regresión Lineal', 'KNN Regressor'])

display(tabla_regresion.round(3))

### Selección del mejor modelo de regresión

Elegiremos el modelo que tenga menor MAE en los datos de prueba, revisando también la diferencia entre entrenamiento y prueba. No basta con que el modelo recuerde los datos que ya conoce: debe funcionar con propiedades que no utilizó para entrenarse.

In [ ]:
tabla_regresion['Diferencia_MAE'] = abs(
    tabla_regresion['MAE_train'] - tabla_regresion['MAE_test']
)

display(tabla_regresion.round(3))

mejor_regresion = tabla_regresion['MAE_test'].idxmin()
print('El modelo con menor error en prueba es:', mejor_regresion)
print('MAE de prueba: USD', round(tabla_regresion.loc[mejor_regresion, 'MAE_test'], 2))
print('RMSE de prueba: USD', round(tabla_regresion.loc[mejor_regresion, 'RMSE_test'], 2))
print('R2 de prueba:', round(tabla_regresion.loc[mejor_regresion, 'R2_test'], 3))

tabla_regresion[['MAE_train', 'MAE_test']].plot(kind='bar', figsize=(8,4))
plt.title('Comparación del MAE en entrenamiento y prueba')
plt.ylabel('Error en dólares')
plt.xlabel('Modelo')
plt.xticks(rotation=0)
plt.show()

### Cross Validation para regresión

Como vimos en clases, la validación cruzada divide los datos de entrenamiento en varias partes. El modelo se entrena y valida varias veces. Revisaremos el promedio y la desviación estándar de los resultados.

Para que la normalización se calcule nuevamente en cada división, utilizamos Pipeline junto con cross_validate.

In [ ]:
# Tomamos una muestra del conjunto de entrenamiento para reducir el tiempo de cálculo
N_CV_REG = min(2500, len(X_train_reg_dummies))
X_cv_reg = X_train_reg_dummies.sample(n=N_CV_REG, random_state=RANDOM_STATE)
y_cv_reg = y_train_reg.loc[X_cv_reg.index]

# Creamos los modelos para Cross Validation
modelo_lineal_cv = Pipeline([
    ('escalado', StandardScaler()),
    ('modelo', LinearRegression())
])

modelo_knn_cv = Pipeline([
    ('escalado', StandardScaler()),
    ('modelo', KNeighborsRegressor(n_neighbors=7))
])

# Aplicamos Cross Validation
cv_lineal = cross_validate(
    modelo_lineal_cv, X_cv_reg, y_cv_reg,
    cv=3,
    scoring='r2',
    return_train_score=True
)

cv_knn = cross_validate(
    modelo_knn_cv, X_cv_reg, y_cv_reg,
    cv=3,
    scoring='r2',
    return_train_score=True
)

print('Regresión Lineal - resultados de cada división:')
print(cv_lineal['test_score'])
print('Promedio:', np.mean(cv_lineal['test_score']))
print('Desviación estándar:', np.std(cv_lineal['test_score']))

print('\\nKNN Regressor - resultados de cada división:')
print(cv_knn['test_score'])
print('Promedio:', np.mean(cv_knn['test_score']))
print('Desviación estándar:', np.std(cv_knn['test_score']))

## 10. Preparación de los datos para clasificación

Ahora predeciremos el tipo de propiedad. La variable property_type ya fue agrupada en cuatro clases: Departamento, Casa, PH y Otros.

Primero convertimos las variables categóricas l2 y l3 mediante One-Hot Encoding. Después normalizamos los atributos numéricos. El escalador se ajusta solo con entrenamiento y se aplica a prueba.

In [ ]:
# Separamos las variables categóricas y numéricas
X_train_clf_dummies = pd.get_dummies(
    X_train_clf,
    columns=features_clf_categoricas,
    drop_first=True
)

X_test_clf_dummies = pd.get_dummies(
    X_test_clf,
    columns=features_clf_categoricas,
    drop_first=True
)

# Igualamos las columnas de entrenamiento y prueba
X_test_clf_dummies = X_test_clf_dummies.reindex(
    columns=X_train_clf_dummies.columns,
    fill_value=0
)

print('Dimensión de X_train:', X_train_clf_dummies.shape)
print('Dimensión de X_test:', X_test_clf_dummies.shape)

# Normalizamos los datos
scaler_clf = StandardScaler()

X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf_dummies)
X_test_clf_scaled = scaler_clf.transform(X_test_clf_dummies)

print('Datos de clasificación preparados correctamente.')

## 11. Modelos de clasificación

Implementaremos los cuatro algoritmos que hemos revisado en clases:

1. Regresión Logística.
2. KNN.
3. SVM.
4. Árbol de Decisión (Decision Tree).

Los cuatro modelos utilizarán los mismos datos de entrenamiento y prueba para que la comparación sea justa.

In [ ]:
# 1. Regresión Logística
modelo_logistico = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
modelo_logistico.fit(X_train_clf_scaled, y_train_clf)

# 2. KNN
modelo_knn = KNeighborsClassifier()
modelo_knn.fit(X_train_clf_scaled, y_train_clf)

# 3. SVM con kernel RBF, como en el ejercicio de Iris
modelo_svm = SVC(kernel='rbf', gamma='auto', probability=True, random_state=RANDOM_STATE)
modelo_svm.fit(X_train_clf_scaled, y_train_clf)

# 4. Árbol de Decisión con parámetros trabajados en clases
modelo_arbol = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    min_samples_leaf=3,
    random_state=100
)
modelo_arbol.fit(X_train_clf_scaled, y_train_clf)

print('Los cuatro modelos fueron entrenados correctamente.')

## 12. Evaluación de los modelos de clasificación

Realizaremos predicciones sobre entrenamiento y prueba. Utilizaremos accuracy, precision, recall, F1 y ROC-AUC. También imprimiremos el classification_report de cada modelo, como en los notebooks de clases.

In [ ]:
ETIQUETAS = ['Departamento', 'Casa', 'PH', 'Otros']

modelos_clasificacion = {
    'Regresión Logística': modelo_logistico,
    'KNN': modelo_knn,
    'SVM': modelo_svm,
    'Decision Tree': modelo_arbol
}

resultados_clasificacion = []
predicciones_test = {}

for nombre, modelo in modelos_clasificacion.items():
    # Predicciones de entrenamiento y prueba
    pred_train = modelo.predict(X_train_clf_scaled)
    pred_test = modelo.predict(X_test_clf_scaled)
    predicciones_test[nombre] = pred_test

    # Probabilidades para calcular ROC-AUC multiclase
    prob_test = modelo.predict_proba(X_test_clf_scaled)

    resultados_clasificacion.append({
        'Modelo': nombre,
        'Accuracy_train': accuracy_score(y_train_clf, pred_train),
        'Accuracy_test': accuracy_score(y_test_clf, pred_test),
        'Precision_macro': precision_score(y_test_clf, pred_test, average='macro', zero_division=0),
        'Recall_macro': recall_score(y_test_clf, pred_test, average='macro', zero_division=0),
        'F1_macro': f1_score(y_test_clf, pred_test, average='macro', zero_division=0),
        'F1_weighted': f1_score(y_test_clf, pred_test, average='weighted', zero_division=0),
        'ROC_AUC': roc_auc_score(
            y_test_clf, prob_test,
            labels=modelo.classes_,
            multi_class='ovr',
            average='macro'
        )
    })

tabla_clasificacion = pd.DataFrame(resultados_clasificacion).set_index('Modelo')
display(tabla_clasificacion.round(3))

### Matriz de confusión y Classification Report

La matriz de confusión permite observar cuántas propiedades fueron clasificadas correctamente y cuáles se confundieron con otra categoría.

Las filas representan la clase real y las columnas la clase predicha. El classification_report muestra precision, recall y F1 para cada clase.

In [ ]:
for nombre, modelo in modelos_clasificacion.items():
    print('\\nModelo:', nombre)

    pred_test = predicciones_test[nombre]

    # Reporte de clasificación
    print(classification_report(
        y_test_clf,
        pred_test,
        labels=ETIQUETAS,
        zero_division=0
    ))

    # Matriz de confusión
    matriz = confusion_matrix(y_test_clf, pred_test, labels=ETIQUETAS)
    print('Matriz de confusión:')
    print(matriz)

    plt.figure(figsize=(6, 4))
    sns.heatmap(
        matriz,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=ETIQUETAS,
        yticklabels=ETIQUETAS
    )
    plt.title('Matriz de confusión - ' + nombre)
    plt.xlabel('Predicción')
    plt.ylabel('Valor real')
    plt.show()

### Cálculo de TP, FP, FN, precision y recall

Para interpretar la clase Otros, tomaremos sus verdaderos positivos (TP), falsos positivos (FP) y falsos negativos (FN) desde la matriz de confusión del árbol de decisión.

- Precision = TP / (TP + FP).
- Recall = TP / (TP + FN).

Esto permite observar cuántas propiedades de tipo Otros reconoce el modelo y cuántas confunde con otra clase.

In [ ]:
# Utilizamos la matriz de confusión del árbol de decisión
matriz_arbol = confusion_matrix(
    y_test_clf,
    predicciones_test['Decision Tree'],
    labels=ETIQUETAS
)

posicion_otros = ETIQUETAS.index('Otros')

TP = matriz_arbol[posicion_otros, posicion_otros]
FP = matriz_arbol[:, posicion_otros].sum() - TP
FN = matriz_arbol[posicion_otros, :].sum() - TP

precision_otros = TP / (TP + FP) if (TP + FP) > 0 else 0
recall_otros = TP / (TP + FN) if (TP + FN) > 0 else 0

print('TP:', TP)
print('FP:', FP)
print('FN:', FN)
print('Precision de Otros:', round(precision_otros, 3))
print('Recall de Otros:', round(recall_otros, 3))

## 13. Balance de clases

En Properati hay muchas más publicaciones de Departamento que de las otras categorías. Por esta razón, el accuracy por sí solo puede entregar una impresión equivocada del rendimiento.

Aplicaremos RandomOverSampler solamente sobre los datos de entrenamiento. Los datos de prueba mantendrán su distribución original. Esta técnica aumenta la cantidad de registros de las clases minoritarias mediante la repetición de observaciones.

In [ ]:
# Revisamos la distribución antes del balance
print('Distribución antes del balance:')
print(y_train_clf.value_counts())

# Aplicamos el balance solamente sobre entrenamiento
balanceador = RandomOverSampler(random_state=RANDOM_STATE)
X_train_balanceado, y_train_balanceado = balanceador.fit_resample(
    X_train_clf_scaled,
    y_train_clf
)

print('\\nDistribución después del balance:')
print(y_train_balanceado.value_counts())

# Entrenamos nuevamente el árbol de decisión
modelo_arbol_balanceado = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    min_samples_leaf=3,
    random_state=100
)

modelo_arbol_balanceado.fit(X_train_balanceado, y_train_balanceado)

# Evaluamos con el conjunto de prueba original
pred_balanceado = modelo_arbol_balanceado.predict(X_test_clf_scaled)

print('\\nClassification Report - Árbol balanceado')
print(classification_report(
    y_test_clf,
    pred_balanceado,
    labels=ETIQUETAS,
    zero_division=0
))

print('Matriz de confusión - Árbol balanceado')
matriz_balanceada = confusion_matrix(
    y_test_clf,
    pred_balanceado,
    labels=ETIQUETAS
)
print(matriz_balanceada)

plt.figure(figsize=(6,4))
sns.heatmap(
    matriz_balanceada,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=ETIQUETAS,
    yticklabels=ETIQUETAS
)
plt.title('Decision Tree con datos balanceados')
plt.xlabel('Predicción')
plt.ylabel('Valor real')
plt.show()

print('F1 macro sin balance:', round(f1_score(
    y_test_clf, predicciones_test['Decision Tree'], average='macro'
), 3))
print('F1 macro con balance:', round(f1_score(
    y_test_clf, pred_balanceado, average='macro'
), 3))
print('Recall de Otros sin balance:', round(recall_score(
    y_test_clf, predicciones_test['Decision Tree'],
    labels=['Otros'], average='macro', zero_division=0
), 3))
print('Recall de Otros con balance:', round(recall_score(
    y_test_clf, pred_balanceado,
    labels=['Otros'], average='macro', zero_division=0
), 3))

## 14. Cross Validation y diagnóstico de generalización

Como en el notebook de Cross Validation del profesor, utilizaremos cross_validate para obtener varios resultados, el promedio y la desviación estándar.

También compararemos el F1 de entrenamiento y prueba. Un resultado muy alto en entrenamiento y mucho menor en prueba puede indicar overfitting. Un resultado bajo en ambos puede indicar underfitting.

In [ ]:
# Usamos una muestra del entrenamiento para reducir el tiempo de cálculo
N_CV_CLF = min(2500, len(X_train_clf_dummies))
X_cv_clf = X_train_clf_dummies.sample(n=N_CV_CLF, random_state=RANDOM_STATE)
y_cv_clf = y_train_clf.loc[X_cv_clf.index]

# Creamos los modelos para la validación cruzada
modelos_cv = {
    'Regresión Logística': Pipeline([
        ('escalado', StandardScaler()),
        ('modelo', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'KNN': Pipeline([
        ('escalado', StandardScaler()),
        ('modelo', KNeighborsClassifier())
    ]),
    'SVM': Pipeline([
        ('escalado', StandardScaler()),
        ('modelo', SVC(kernel='rbf', gamma='auto'))
    ]),
    'Decision Tree': Pipeline([
        ('escalado', StandardScaler()),
        ('modelo', DecisionTreeClassifier(
            criterion='gini', max_depth=5,
            min_samples_leaf=3, random_state=100
        ))
    ])
}

resultados_cv = []

for nombre, modelo in modelos_cv.items():
    cv_resultado = cross_validate(
        modelo,
        X_cv_clf,
        y_cv_clf,
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        scoring='f1_macro',
        return_train_score=True
    )

    resultados_cv.append({
        'Modelo': nombre,
        'F1_train_promedio': np.mean(cv_resultado['train_score']),
        'F1_validacion_promedio': np.mean(cv_resultado['test_score']),
        'Desviacion_estandar': np.std(cv_resultado['test_score'])
    })

tabla_cv = pd.DataFrame(resultados_cv).set_index('Modelo')
display(tabla_cv.round(3))

print('Los resultados muestran el promedio y la desviación estándar de las divisiones.')

### Comparación de overfitting y underfitting en Decision Tree

El profesor trabaja con el parámetro max_depth para controlar la profundidad del árbol. Compararemos un árbol simple, uno con profundidad 5 y uno sin límite de profundidad.

In [ ]:
# Árbol simple
arbol_simple = DecisionTreeClassifier(max_depth=1, random_state=100)
arbol_simple.fit(X_train_clf_scaled, y_train_clf)

# Árbol utilizado en la comparación principal
arbol_profundidad_5 = DecisionTreeClassifier(
    criterion='gini', max_depth=5,
    min_samples_leaf=3, random_state=100
)
arbol_profundidad_5.fit(X_train_clf_scaled, y_train_clf)

# Árbol sin límite de profundidad
arbol_sin_limite = DecisionTreeClassifier(random_state=100)
arbol_sin_limite.fit(X_train_clf_scaled, y_train_clf)

arboles = {
    'Árbol simple (profundidad 1)': arbol_simple,
    'Árbol profundidad 5': arbol_profundidad_5,
    'Árbol sin límite': arbol_sin_limite
}

resultados_arboles = []

for nombre, modelo in arboles.items():
    pred_train = modelo.predict(X_train_clf_scaled)
    pred_test = modelo.predict(X_test_clf_scaled)

    resultados_arboles.append({
        'Modelo': nombre,
        'F1_train': f1_score(y_train_clf, pred_train, average='macro', zero_division=0),
        'F1_test': f1_score(y_test_clf, pred_test, average='macro', zero_division=0)
    })

tabla_arboles = pd.DataFrame(resultados_arboles).set_index('Modelo')
tabla_arboles['Diferencia'] = tabla_arboles['F1_train'] - tabla_arboles['F1_test']
display(tabla_arboles.round(3))

## 15. Selección del mejor clasificador y conclusiones

Para elegir el modelo no observaremos solamente el accuracy. Dado que las clases están desbalanceadas, daremos importancia al F1 macro, que considera por igual a Departamento, Casa, PH y Otros.

También revisaremos la validación cruzada y la diferencia entre entrenamiento y prueba.

**Ventajas y desventajas de las métricas:**

- Accuracy: es fácil de interpretar, pero puede ocultar errores en las clases minoritarias.
- Precision: indica qué proporción de las predicciones de una clase fue correcta. Puede bajar si el modelo genera más falsos positivos.
- Recall: indica qué proporción de los casos reales de una clase fue encontrada. Puede bajar cuando el modelo deja casos sin detectar.
- F1: combina precision y recall. Es útil cuando ambas importan, pero no muestra por separado cuál es menor.
- ROC-AUC: permite evaluar la capacidad de separar las clases usando probabilidades. Para este problema se calcula con estrategia One-vs-Rest.
- Macro: todas las clases tienen el mismo peso. Weighted: cada clase pesa según la cantidad de registros que contiene.

En el negocio, confundir una Casa con un Departamento puede mostrar una publicación en una categoría incorrecta. No reconocer PH u Otros puede ocultar propiedades de categorías menos frecuentes.

In [ ]:
# Agregamos las métricas de validación cruzada a la tabla de clasificación
tabla_final = tabla_clasificacion.join(tabla_cv)

# Calculamos la diferencia entre el resultado de entrenamiento y prueba
tabla_final['Diferencia_accuracy'] = (
    tabla_final['Accuracy_train'] - tabla_final['Accuracy_test']
)

display(tabla_final.sort_values('F1_macro', ascending=False).round(3))

# Seleccionamos el modelo con mejor F1 macro en prueba
mejor_clasificador = tabla_final['F1_macro'].idxmax()

print('El mejor clasificador según F1 macro en prueba es:', mejor_clasificador)
print('F1 macro:', round(tabla_final.loc[mejor_clasificador, 'F1_macro'], 3))
print('Recall macro:', round(tabla_final.loc[mejor_clasificador, 'Recall_macro'], 3))
print('ROC-AUC:', round(tabla_final.loc[mejor_clasificador, 'ROC_AUC'], 3))
print('F1 macro en validación cruzada:', round(
    tabla_final.loc[mejor_clasificador, 'F1_validacion_promedio'], 3
))
print('Desviación estándar:', round(
    tabla_final.loc[mejor_clasificador, 'Desviacion_estandar'], 3
))

# Mostramos la comparación de F1 macro
tabla_final['F1_macro'].sort_values().plot(kind='barh', figsize=(8,4))
plt.title('Comparación de F1 macro de los clasificadores')
plt.xlabel('F1 macro')
plt.ylabel('Modelo')
plt.show()

### Conclusiones del trabajo

El modelo de regresión se selecciona considerando el error en dólares sobre los datos de prueba. El modelo de clasificación se selecciona considerando F1 macro, porque interesa reconocer las cuatro categorías y no solamente la clase mayoritaria.

Los resultados de balanceo permiten observar si mejora el reconocimiento de las clases minoritarias y si esa mejora afecta al resultado general.

Los resultados obtenidos corresponden a la muestra utilizada de Properati. Antes de utilizar un modelo en un sistema real, sería necesario probarlo con publicaciones nuevas y revisar la calidad de los datos.

In [ ]:
print('CONCLUSIONES BASADAS EN LOS RESULTADOS')

print('\\nRegresión:')
print('Modelo seleccionado:', mejor_regresion)
print('MAE en prueba: USD', round(tabla_regresion.loc[mejor_regresion, 'MAE_test'], 2))
print('R2 en prueba:', round(tabla_regresion.loc[mejor_regresion, 'R2_test'], 3))

print('\\nClasificación:')
print('Modelo seleccionado:', mejor_clasificador)
print('F1 macro en prueba:', round(tabla_final.loc[mejor_clasificador, 'F1_macro'], 3))
print('Recall macro en prueba:', round(tabla_final.loc[mejor_clasificador, 'Recall_macro'], 3))
print('ROC-AUC:', round(tabla_final.loc[mejor_clasificador, 'ROC_AUC'], 3))

print('\\nBalance de clases:')
print('F1 macro sin balance:', round(f1_score(
    y_test_clf, predicciones_test['Decision Tree'], average='macro'
), 3))
print('F1 macro con balance:', round(f1_score(
    y_test_clf, pred_balanceado, average='macro'
), 3))

print('\\nOverfitting y underfitting:')
display(tabla_arboles.round(3))

## Checklist de la evaluación

| Indicador | Contenido |
|---|---|
| IE 1 | Target continuo: price |
| IE 2 | Regresión lineal y KNN Regressor |
| IE 3 | MAE, RMSE y R² en entrenamiento y prueba |
| IE 4 | Comparación y selección del regresor |
| IE 5 | Diferencia entre regresión y clasificación |
| IE 6 | Correlación Pearson, Spearman y heatmap |
| IE 7 | One-Hot Encoding y normalización |
| IE 8 | Justificación de las características seleccionadas |
| IE 9 | Regresión Logística, KNN, SVM y Decision Tree |
| IE 10 | Comparación de entrenamiento, prueba y Cross Validation |
| IE 11 | Classification Report, métricas y ROC-AUC |
| IE 12 | Matriz de confusión y cálculo de TP, FP y FN |
| IE 13 | Balance de clases solo en entrenamiento |
| IE 14 | Comparación y selección del clasificador |
| IE 15 | Ventajas y desventajas de las métricas |

El notebook debe ejecutarse desde la primera celda hasta la última. Las conclusiones deben revisarse junto con los resultados obtenidos.